# 🔧 Notebook 3 – Feature Engineering

**Project:** EcoType – Forest Cover Type Prediction  
**Objective:** Create new derived features, encode categorical variables, and save the processed data for modeling.

---

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
df = pd.read_csv("../data/processed/capped_forest_cover_data.csv")
print(f"✅ Loaded cleaned data: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")

✅ Loaded cleaned data: 145,890 rows × 13 columns
Columns: ['Elevation', 'Aspect', 'Slope', 'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology', 'Horizontal_Distance_To_Roadways', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm', 'Horizontal_Distance_To_Fire_Points', 'Cover_Type', 'Wilderness_Area', 'Soil_Type']


---
## 4️⃣ Feature Engineering

Create derived columns to improve model interpretability and predictive power.

### 4.1 Hillshade Difference (9am – 3pm)
Captures the directional sunlight change throughout the day.

In [2]:
df["Hillshade_diff_9am_3pm"] = df["Hillshade_9am"] - df["Hillshade_3pm"]
print(f"✅ Created 'Hillshade_diff_9am_3pm'  |  Range: [{df['Hillshade_diff_9am_3pm'].min()}, {df['Hillshade_diff_9am_3pm'].max()}]")

✅ Created 'Hillshade_diff_9am_3pm'  |  Range: [-46.5, 190.0]


### 4.2 Mean Hillshade
Average illumination across three time points.

In [3]:
df["Hillshade_mean"] = df[["Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm"]].mean(axis=1)
print(f"✅ Created 'Hillshade_mean'  |  Range: [{df['Hillshade_mean'].min():.2f}, {df['Hillshade_mean'].max():.2f}]")

✅ Created 'Hillshade_mean'  |  Range: [139.83, 213.67]


### 4.3 Elevation × Slope Interaction
Captures the combined effect of elevation and terrain steepness.

In [4]:
df["Elevation_Slope"] = df["Elevation"] * df["Slope"]
print(f"✅ Created 'Elevation_Slope'  |  Range: [{df['Elevation_Slope'].min():.1f}, {df['Elevation_Slope'].max():.1f}]")

✅ Created 'Elevation_Slope'  |  Range: [0.0, 91516.5]


### 4.4 Hydrology Distance Magnitude
Euclidean distance combining horizontal and vertical distances to water.

In [5]:
df["Hydrology_Distance_Mag"] = (
    df['Horizontal_Distance_To_Hydrology']**2 +
    df['Vertical_Distance_To_Hydrology']**2
) ** 0.5
print(f"✅ Created 'Hydrology_Distance_Mag'  |  Range: [{df['Hydrology_Distance_Mag'].min():.2f}, {df['Hydrology_Distance_Mag'].max():.2f}]")

✅ Created 'Hydrology_Distance_Mag'  |  Range: [0.00, 771.42]


### 4.5 Road-to-Fire Distance Ratio
Ratio of distance to roadways vs. distance to wildfire ignition points.

In [6]:
df['Road_Fire_Distance_Ratio'] = (
    df['Horizontal_Distance_To_Roadways'] /
    (df['Horizontal_Distance_To_Fire_Points'] + 1)  # +1 to avoid division by zero
)
print(f"✅ Created 'Road_Fire_Distance_Ratio'  |  Range: [{df['Road_Fire_Distance_Ratio'].min():.4f}, {df['Road_Fire_Distance_Ratio'].max():.4f}]")

✅ Created 'Road_Fire_Distance_Ratio'  |  Range: [0.0000, 4818.0000]


### 4.6 Verify Engineered Features

In [7]:
# Check for NaN or Inf values in numeric columns
num_df = df.select_dtypes(include=[np.number])
print(f"NaN values: {num_df.isna().sum().sum()}")
print(f"Inf values: {np.isinf(num_df).sum().sum()}")
print(f"\nFinal shape: {df.shape}")
print(f"\nAll columns ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"  {i:2d}. {col} ({df[col].dtype})")

NaN values: 0
Inf values: 0

Final shape: (145890, 18)

All columns (18):
   1. Elevation (float64)
   2. Aspect (int64)
   3. Slope (int64)
   4. Horizontal_Distance_To_Hydrology (float64)
   5. Vertical_Distance_To_Hydrology (int64)
   6. Horizontal_Distance_To_Roadways (int64)
   7. Hillshade_9am (float64)
   8. Hillshade_Noon (int64)
   9. Hillshade_3pm (int64)
  10. Horizontal_Distance_To_Fire_Points (int64)
  11. Cover_Type (object)
  12. Wilderness_Area (int64)
  13. Soil_Type (int64)
  14. Hillshade_diff_9am_3pm (float64)
  15. Hillshade_mean (float64)
  16. Elevation_Slope (float64)
  17. Hydrology_Distance_Mag (float64)
  18. Road_Fire_Distance_Ratio (float64)


---
## 4.7 Encode Target Variable & Save Encoder
Encode the string Cover_Type labels to integers using LabelEncoder. **The encoder is saved** for later use in the Streamlit app (inverse transform).

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Cover_Type'] = le.fit_transform(df['Cover_Type'])

# Save the encoder for inference (critical for Streamlit inverse transform)
joblib.dump(le, "../models/label_encoder.pkl")

print("✅ Cover_Type encoded:")
print(f"   Classes: {list(le.classes_)}")
print(f"   Mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"\n✅ LabelEncoder saved to: ../models/label_encoder.pkl")

✅ Cover_Type encoded:
   Classes: ['Aspen', 'Cottonwood/Willow', 'Douglas-fir', 'Krummholz', 'Lodgepole Pine', 'Ponderosa Pine', 'Spruce/Fir']
   Mapping: {'Aspen': np.int64(0), 'Cottonwood/Willow': np.int64(1), 'Douglas-fir': np.int64(2), 'Krummholz': np.int64(3), 'Lodgepole Pine': np.int64(4), 'Ponderosa Pine': np.int64(5), 'Spruce/Fir': np.int64(6)}

✅ LabelEncoder saved to: ../models/label_encoder.pkl


---
## 4.8 Reorder & Save Engineered Data

In [9]:
# Reorder columns for clarity
column_order = [
    'Elevation', 'Aspect', 'Slope',
    'Horizontal_Distance_To_Hydrology', 'Vertical_Distance_To_Hydrology',
    'Horizontal_Distance_To_Roadways',
    'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm',
    'Horizontal_Distance_To_Fire_Points',
    'Wilderness_Area', 'Soil_Type',
    'Hillshade_diff_9am_3pm', 'Hillshade_mean',
    'Elevation_Slope', 'Hydrology_Distance_Mag',
    'Road_Fire_Distance_Ratio',
    'Cover_Type'
]
df = df[column_order]

# Save
df.to_csv("../data/processed/feature_engineered_forest_cover_data.csv", index=False)
print(f"✅ Feature-engineered data saved: {df.shape}")
df.info()

✅ Feature-engineered data saved: (145890, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145890 entries, 0 to 145889
Data columns (total 18 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   Elevation                           145890 non-null  float64
 1   Aspect                              145890 non-null  int64  
 2   Slope                               145890 non-null  int64  
 3   Horizontal_Distance_To_Hydrology    145890 non-null  float64
 4   Vertical_Distance_To_Hydrology      145890 non-null  int64  
 5   Horizontal_Distance_To_Roadways     145890 non-null  int64  
 6   Hillshade_9am                       145890 non-null  float64
 7   Hillshade_Noon                      145890 non-null  int64  
 8   Hillshade_3pm                       145890 non-null  int64  
 9   Horizontal_Distance_To_Fire_Points  145890 non-null  int64  
 10  Wilderness_Area                     145890 non

---
## 📝 Feature Engineering Summary

| New Feature | Formula | Purpose |
|------------|---------|--------|
| `Hillshade_diff_9am_3pm` | Hillshade_9am - Hillshade_3pm | Sunlight direction change |
| `Hillshade_mean` | Mean of 3 hillshade values | Average illumination |
| `Elevation_Slope` | Elevation * Slope | Terrain interaction |
| `Hydrology_Distance_Mag` | sqrt(H^2 + V^2) | True distance to water |
| `Road_Fire_Distance_Ratio` | Road_Dist / (Fire_Dist + 1) | Relative accessibility |
| `Cover_Type` (encoded) | LabelEncoder | Numeric target for modeling |

> **Note:** The LabelEncoder is saved to `models/label_encoder.pkl` for use during Streamlit inference.